In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
from deep_bf.config_registery import BeamformerConfig, ResamplerConfig, ApodConfig, DataSizeConfig, CompoundingConfig
from deep_bf.config_registery.path_center import PathCenter
from deep_bf.data_handler import DataLoader
from deep_bf.constants.bf import BeamformerType, CompooundingType

dasC = BeamformerConfig(23, BeamformerType.DAS, {"batch_size":128})

mvC = BeamformerConfig(28, BeamformerType.MV, {"batch_size":1, "z_chunk": 512, "L":16, "diagonal_loading":1e-3, "eps":1e-10})


dmasC = BeamformerConfig(
    28,
    BeamformerType.FDMAS,
    {
        "batch_size": 128,
        "BW": 0.6,
        "bp_window": "hanning",
        "eps": 1e-8,
        # nueva logica robusta
        "fallback_mode": "auto",          # intenta 2f0, si no puede usa f0
        "allow_asymmetric": True,         # permite banda recortada por Nyquist
        "nyquist_margin": 0.995,          # evita usar centro pegado a Nyquist
        "min_band_bins": 4,               # evita filtros casi vacios
        "strict_if_no_valid_band": False, # si no hay banda valida, no rompe
    }
)

cfC = BeamformerConfig(24, BeamformerType.CF, {"batch_size":128, "eps":1e-8})
imapC = BeamformerConfig(25, BeamformerType.IMAP, {"batch_size":128, "num_iters":5, "eps":1e-8})
sr1C = BeamformerConfig(26, BeamformerType.SR1, {"batch_size":128, "num_iters":20, "lam": 1e-3, "step":1.0, "eps":1e-8})
sr2C = BeamformerConfig(27, BeamformerType.SR2, {
    "batch_size": 128,
    "num_iters": 15,
    "lam": 1e-4,
    "tau": 0.2,
    "sigma": 0.2,
    "theta": 1.0,
    "eps": 1e-8,
    "J": 2,
    "mode": "symmetric",
})

gsC = ResamplerConfig(
    24,
    "GridSample",
    {"align_corners": False, "mode": "bilinear", "padding_mode": "border"}
)

nC = ApodConfig(
    23,
    "None",
    {}
)

d1C = DataSizeConfig(11, 2048, 256, -1)
d2C = DataSizeConfig(12, 1024, 256, -1)
d3C = DataSizeConfig(13, 512, 256, -1)
d4C = DataSizeConfig(14, 256, 256, -1)
d5C = DataSizeConfig(15, -1, -1, -1)

cpC = CompoundingConfig(1, CompooundingType.CPWC_MEAN, {})
acpC = CompoundingConfig(2, CompooundingType.ANGULAR_CPWC_SHORT_PULSE, {"kind": "hanning"})
acp2C = CompoundingConfig(2, CompooundingType.ANGULAR_CPWC_PARAXIAL, {"kind": "hanning"})
aedtC = CompoundingConfig(3, CompooundingType.CPWC_EDT, {"alpha": 0.001, "eps": 1e-8})


In [ ]:
from deep_bf.config_registery import ConfigRegisteryCenter

with ConfigRegisteryCenter() as c:
    c.add_beamformer(dasC)
    c.add_beamformer(mvC)
    c.add_beamformer(dmasC)
    c.add_beamformer(cfC)
    c.add_beamformer(imapC)
    c.add_beamformer(sr1C)
    c.add_beamformer(sr2C)


In [5]:
dCs = [
    DataSizeConfig(-1, 512, 256, 2300),
    DataSizeConfig(-1, 256, 256, 2300),
    DataSizeConfig(-1, -1, -1, 2300),
]

with ConfigRegisteryCenter() as c:
    for dC in dCs:
        c.add_data_size(dC)

In [6]:
cpCs = [
    CompoundingConfig(1, CompooundingType.CPWC_MEAN, {}),
    CompoundingConfig(2, CompooundingType.ANGULAR_CPWC_PARAXIAL, {"kind": "hanning"}),
    CompoundingConfig(3, CompooundingType.CPWC_EDT, {"alpha": 0.001, "eps": 1e-8})
]

with ConfigRegisteryCenter() as c:
    for cpC in cpCs:
        c.add_compounding(cpC)

In [ ]:
bfSs = [
    {"data_type_config_id": 0, "beamformer_config_id": 0, "resampler_config_id": 0, "compounding_config_id": 0, "apod_config_id": 0}
]

with ConfigRegisteryCenter() as c:
    for bfS in bfSs:
        c.add_beamformer_setup()